Preprocessing, keypoints of surdobot videos.

In [1]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Output format change: .npz instead of bare .npy
# ---------------------------------------------------------------------------
# Each file now stores three arrays instead of one:
#   keypoints     : (T, 2, 21, 2) float32   -- same as before
#   frame_numbers : (T,)          int64     -- the ABSOLUTE source video
#                                              frame index for each row.
#                                              This is what makes merging
#                                              across shards correct, and
#                                              what lets you align a row
#                                              here with the frame indices
#                                              VideoMAE sampled from the
#                                              raw video (decord/opencv
#                                              index into the same space).
#   hand_valid    : (T, 2)        bool      -- True where that hand was
#                                              actually detected in that
#                                              frame, independent of
#                                              whether the coordinate
#                                              values happen to be zero.
#                                              (see note below on why this
#                                              is needed once keypoints are
#                                              wrist-relative)
# ---------------------------------------------------------------------------


def process_single_video_group(video_id, new_group, coord_cols, output_dir):
    """
    Processes or updates the keypoint record for a specific video ID,
    correctly merging split data across CSV shards by absolute frame
    number rather than dense array position.
    """
    target_path = output_dir / f"{video_id}.npz"

    clean_group = new_group.dropna(subset=['frame', 'hand'])
    if clean_group.empty:
        return

    new_frames = clean_group['frame'].values
    new_hands = clean_group['hand'].values
    new_coords_raw = clean_group[coord_cols].fillna(0.0).values.astype(np.float32)

    # registry keyed by (absolute_frame_number, hand_idx) -> (21,2) matrix
    frame_registry = {}

    # ── CASE 1: reload any previously merged shards for this video ──
    if target_path.exists():
        try:
            existing = np.load(target_path)
            existing_kp = existing['keypoints']            # (T_old, 2, 21, 2)
            existing_frame_numbers = existing['frame_numbers']  # (T_old,)
            existing_valid = existing['hand_valid']         # (T_old, 2)

            for t_idx, abs_frame in enumerate(existing_frame_numbers):
                for h_idx in (0, 1):
                    if existing_valid[t_idx, h_idx]:
                        # keyed by the REAL frame number this time, not t_idx
                        frame_registry[(int(abs_frame), h_idx)] = existing_kp[t_idx, h_idx, :, :]
        except Exception as e:
            print(f"  [warn] could not reload existing file for {video_id} ({e}); rebuilding from scratch")

    # ── CASE 2: merge in this shard's rows ──
    for i in range(len(new_frames)):
        f_num = int(new_frames[i])
        h_side = new_hands[i]
        h_idx = 0 if h_side == 'Left' else 1
        frame_registry[(f_num, h_idx)] = new_coords_raw[i].reshape(21, 2)

    if not frame_registry:
        return

    # ── Final packaging, indexed by absolute frame number ──
    all_unique_frames = sorted(set(k[0] for k in frame_registry.keys()))
    total_frames = len(all_unique_frames)
    frame_to_idx = {frame: i for i, frame in enumerate(all_unique_frames)}

    tensor = np.zeros((total_frames, 2, 21, 2), dtype=np.float32)
    valid = np.zeros((total_frames, 2), dtype=bool)

    for (f_num, h_idx), joint_matrix in frame_registry.items():
        t_idx = frame_to_idx[f_num]
        tensor[t_idx, h_idx, :, :] = joint_matrix
        valid[t_idx, h_idx] = True

    frame_numbers_arr = np.array(all_unique_frames, dtype=np.int64)

    np.savez(
        target_path,
        keypoints=tensor,
        frame_numbers=frame_numbers_arr,
        hand_valid=valid,
    )


def process_all_shards(csv_dir, output_dir):
    csv_dir = Path(csv_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_files = sorted(glob.glob(str(csv_dir / "krsl_*_dataset*")))

    if not csv_files:
        print(f"No CSV shard files found in '{csv_dir}'. Check your file names!")
        return

    print(f"Found {len(csv_files)} sharded CSV partitions to process.")

    coord_cols = []
    for i in range(1, 22):
        coord_cols.extend([f'x{i}', f'y{i}'])

    for shard_idx, file_path in enumerate(csv_files):
        print(f"\n[Shard {shard_idx + 1}/{len(csv_files)}] Reading: {os.path.basename(file_path)}")

        df = pd.read_csv(file_path)
        df = df.dropna(subset=['filename'])
        df['video_id'] = df['filename'].apply(lambda x: Path(x).parts[-2].replace('.mp4', ''))

        grouped = df.groupby('video_id')

        for video_id, group in tqdm(grouped, desc="Processing videos in shard"):
            process_single_video_group(video_id, group, coord_cols, output_dir)

        del df
        del grouped

    print(f"\nFinished. All shards compiled. .npz files saved in: '{output_dir}'")


# ---------------------------------------------------------------------------
# Helper for the VideoMAE fusion step: given the frame indices VideoMAE
# sampled from the raw video (e.g. via decord's uniform sampling), pull the
# matching rows out of this video's keypoint file.
# ---------------------------------------------------------------------------

def align_keypoints_to_sampled_frames(npz_path, sampled_frame_indices):
    """
    Returns a (len(sampled_frame_indices), 2, 21, 2) array aligned to the
    exact frame indices used for VideoMAE sampling, plus a matching
    (len(sampled_frame_indices), 2) validity mask.

    Frames with no keypoint detection at that exact index (common if the
    keypoint extractor and VideoMAE sampler land on different frames)
    fall back to the nearest frame that DOES have keypoint data, since
    hand pose changes slowly relative to typical frame gaps. If no
    keypoint data exists at all for the clip, returns zeros / all-invalid.
    """
    data = np.load(npz_path)
    kp = data['keypoints']
    frame_numbers = data['frame_numbers']
    valid = data['hand_valid']

    out_kp = np.zeros((len(sampled_frame_indices), 2, 21, 2), dtype=np.float32)
    out_valid = np.zeros((len(sampled_frame_indices), 2), dtype=bool)

    if len(frame_numbers) == 0:
        return out_kp, out_valid

    for i, target_frame in enumerate(sampled_frame_indices):
        nearest_idx = int(np.argmin(np.abs(frame_numbers - target_frame)))
        out_kp[i] = kp[nearest_idx]
        out_valid[i] = valid[nearest_idx]

    return out_kp, out_valid



In [3]:
CSV_SHARDS_DIRECTORY = "/run/media/gygy/T7/KRSL_OnlineSchool/hs_labeled/"
NP_OUTPUT_DIRECTORY = "/run/media/gygy/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints"

process_all_shards(CSV_SHARDS_DIRECTORY, NP_OUTPUT_DIRECTORY)

Found 22 sharded CSV partitions to process.

[Shard 1/22] Reading: krsl_07_11_[0-500k]_dataset.csv


Processing videos in shard: 100%|██████████| 1347/1347 [01:18<00:00, 17.14it/s]



[Shard 2/22] Reading: krsl_07_11_[1.5m-end]_dataset.csv


Processing videos in shard: 100%|██████████| 2062/2062 [01:54<00:00, 18.00it/s]



[Shard 3/22] Reading: krsl_07_11_-1.5m]_dataset.csv


...


[Shard 21/22] Reading: krsl_27_10_-2.5mil]_dataset.csv


Processing videos in shard: 100%|██████████| 1341/1341 [01:59<00:00, 11.19it/s]



[Shard 22/22] Reading: krsl_27_10_[500k-1mil]_dataset.csv


Processing videos in shard: 100%|██████████| 1328/1328 [01:46<00:00, 12.44it/s]


Finished. All shards compiled. .npz files saved in: '/run/media/gygy/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints'


Preprocessing for annotations

In [5]:
import numpy as np
import pandas as pd
import glob 

csv_files = glob.glob("/run/media/gygy/T7/KRSL_OnlineSchool/KRSL_OnlineSchool_final_dataset/Surdobot_Annotations/*.csv")
df_list = [pd.read_csv(file) for file in csv_files]
# csv_files
combined_df = pd.concat(df_list, ignore_index=True)


In [6]:
combined_df.head()

,ID,Video,Username,Gloss,Time spend,Created at
0,1221,07_11/resized_cropped_18_11_11_13_01-3-of-30.mp4,Ilona,Это группа он . Карбодиная группа это с два ра...,229,2021-03-11 08:09:56.451535+00:00
1,53,07_11/resized_cropped_09_12_10_11_01-6-of-27.mp4,katiya,Сегодня учить будет новый () предложение когда...,104,2021-03-12 18:01:04.269018+00:00
2,14030,2020/resized_cropped_23_12_8_03_03-22-of-27.mp4,katiya,Проверить найти ответ русские 5 метров от горо...,146,2021-03-31 17:02:59.528263+00:00
3,581,07_11/resized_cropped_09_12_7_13_03-11-of-22.mp4,Ilona,Оформление путь был как лартовский это связь б...,223,2021-03-11 14:03:21.036307+00:00
4,2591,07_11/resized_cropped_20_11_9_12_01-9-of-30.mp4,Vova,месяц температура меняются половина север и юг...,269,2021-03-01 06:25:30.794895+00:00


In [21]:
"""
Surdobot annotation parser.

Reads annotation CSVs from Surdobot_Annotations/ and matches each row to
its corresponding video clip in Surdobot_VideoBase/.

Each annotation row contains:
  - A clip identifier (filename or ID)
  - An unordered bag of gloss labels (space/comma separated)

NOTE: Surdobot transcripts (where they exist) live in a separate folder,
not as a CSV column, and are NOT wired up here. Decision: since Surdobot
already has gloss bag labels as its primary supervision signal, the text
modality is not required to train on Surdobot — it matters far more for
Elarna (which has no gloss labels at all). Dropped for now to keep the
pipeline simple; revisit if/when the core MIL/CTC loop is proven to work
and you want to explore whether transcripts add anything on top of the
gloss bags. See elarna_parser.py for the folder-based transcript lookup
pattern (_find_transcript) if you want to wire this up later.

Output per clip:
  {
    "clip_id":        str,
    "video_path":     str,
    "gloss_bag":      List[str],       # unordered, de-duplicated
    "has_audio":      False,           # Surdobot never has audio
    "duration_sec":   float,
    "split":          "train" | "val" | "test"
  }
"""

import random
from pathlib import Path
from typing import Optional
import pandas as pd
import re

from load_config import load_config
from common import get_logger, probe_video, video_id_from_path

logger = get_logger(__name__)


# ─────────────────────────────────────────────
#  Annotation CSV column names
#  Adjust these to match your actual CSV headers
# ─────────────────────────────────────────────

# Try multiple possible column name conventions automatically
_CLIP_ID_COLS = ["Video"]
_GLOSS_COLS   = ["gloss"]


def _find_col(df_cols: list[str], candidates: list[str]) -> Optional[str]:
    """Return the first matching column name from candidates (case-insensitive)."""
    lower = {c.lower(): c for c in df_cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


def _parse_gloss_cell(cell: str) -> list[str]:
    """
    Advanced gloss parser designed to filter out classroom/textbook noise
    like math equations, raw numbers, and alphabet dumps.
    """
    if not isinstance(cell, str) or not cell.strip():
        return []
    
    # Replace common delimiters with spaces
    for delim in [",", ";", "|", "\t", "\n"]:
        cell = cell.replace(delim, " ")
        
    cell = cell.upper()
    raw_tokens = cell.split()
    clean_tokens = []
    
    # Quick filter for math operators frequently found in formulas
    math_chars = set("+-*/=")

    for t in raw_tokens:
        # 1. Skip if it contains explicit math operators
        if any(char in math_chars for char in t):
            continue
            
        # 2. Strip leading/trailing non-alphanumeric clutter
        t_clean = re.sub(r'^[^\w\s\d]+|[^\w\s\d]+$', '', t)
        
        # 3. Skip if it's completely empty or purely numeric (e.g., "180", "0")
        if not t_clean or t_clean.isdigit():
            continue
            
        # 4. Filter out mixed alphanumeric noise (e.g., "0+У" or "2Х")
        # Valid sign language glosses are almost always purely alphabetical strings
        if any(c.isdigit() for c in t_clean) and any(c.isalpha() for c in t_clean):
            continue
            
        # 5. Keep it if it's a valid string of characters
        if any(c.isalpha() for c in t_clean):
            clean_tokens.append(t_clean)
            
    return list(dict.fromkeys(clean_tokens))

def _find_video_file(video_root: Path, clip_id: str) -> Optional[Path]:
    """
    Locate a video file given a clip_id path (e.g., 'subfolder/video.mp4').
    """
    # 1. Try treating clip_id as a direct relative path under video_root
    direct_path = video_root / clip_id
    if direct_path.exists():
        return direct_path

    # 2. Fallback: If the CSV path missing the extension, try adding them
    for ext in [".mp4", ".avi", ".mkv", ".mov", ".webm"]:
        if not direct_path.suffix: # if it doesn't have an extension yet
            path_with_ext = video_root / f"{clip_id}{ext}"
            if path_with_ext.exists():
                return path_with_ext
                
    # 3. Last resort: just look for the filename itself anywhere in the folder
    filename = Path(clip_id).name
    for ext in [".mp4", ".avi", ".mkv", ".mov", ".webm"]:
        stem = Path(filename).stem
        matches = list(video_root.rglob(f"{stem}{ext}"))
        if matches:
            return matches[0]

    return None


def merge_duplicate_clips(records: list[dict]) -> list[dict]:
    """
    Merge multiple annotation rows that share the same clip_id (i.e. the same
    video annotated by different annotators) into a single record per clip.

    Strategy: SIMPLE UNION — the merged gloss_bag is the union of all
    annotators' gloss bags for that clip_id. No per-gloss agreement weighting
    is applied (see conversation: chose simplicity over confidence-weighting).

    Must be called BEFORE train/val/test split assignment, so that all
    annotations of the same clip end up in the same split.

    Args:
        records: List of per-row clip records, possibly with duplicate clip_ids

    Returns:
        List of records with one entry per unique clip_id.
    """
    by_clip: dict[str, list[dict]] = {}
    for rec in records:
        by_clip.setdefault(rec["clip_id"], []).append(rec)

    merged = []
    n_merged_clips = 0
    n_total_dupe_rows = 0
    agreement_log = []

    for clip_id, group in by_clip.items():
        if len(group) == 1:
            merged.append(group[0])
            continue

        # Multiple annotators for this clip — union the gloss bags
        n_merged_clips += 1
        n_total_dupe_rows += len(group) - 1

        union_bag = []
        seen = set()
        for rec in group:
            for g in rec["gloss_bag"]:
                if g not in seen:
                    seen.add(g)
                    union_bag.append(g)

        base = group[0]   # reuse video_path / duration / source from first row
        merged_record = {
            **base,
            "gloss_bag":      union_bag,
            "num_annotators": len(group),
        }
        merged.append(merged_record)

        # Log agreement info for visibility (not used for weighting, just diagnostics)
        bag_sizes = [len(r["gloss_bag"]) for r in group]
        overlap = len(set.intersection(*[set(r["gloss_bag"]) for r in group]))
        agreement_log.append((clip_id, len(group), bag_sizes, len(union_bag), overlap))

    logger.info(f"Merged {n_merged_clips} clips that had multiple annotator rows "
                f"({n_total_dupe_rows} extra rows folded in)")

    if agreement_log:
        avg_overlap_ratio = sum(
            ov / max(1, max(sizes)) for _, _, sizes, _, ov in agreement_log
        ) / len(agreement_log)
        logger.info(f"  Avg annotator agreement (overlap/max_bag_size): "
                    f"{avg_overlap_ratio:.2%}")
        # Show a few examples for sanity-checking
        for clip_id, n_ann, sizes, union_size, overlap in agreement_log[:5]:
            logger.debug(f"  [{clip_id}] {n_ann} annotators, bag sizes={sizes}, "
                        f"union={union_size}, overlap={overlap}")

    return merged


def parse_surdobot(
    annotations_dir: str | Path,
    video_root: str | Path,
    split_ratios: tuple[float, float, float] = (0.80, 0.10, 0.10),
    seed: int = 42,
) -> list[dict]:
    """
    Parse all Surdobot annotation CSVs and return a list of clip records.

    Args:
        annotations_dir: Path to Surdobot_Annotations/
        video_root:      Path to Surdobot_VideoBase/
        split_ratios:    (train, val, test) fractions — must sum to 1.0
        seed:            Random seed for reproducible splits

    Returns:
        List of clip record dicts, each enriched with split assignment.
    """
    annotations_dir = Path(annotations_dir)
    video_root      = Path(video_root)

    assert abs(sum(split_ratios) - 1.0) < 1e-6, "Split ratios must sum to 1.0"

    csv_files = sorted(annotations_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {annotations_dir}")
    logger.info(f"Found {len(csv_files)} annotation CSV(s) in {annotations_dir}")

    records = []
    missing_videos = 0
    total_rows = 0

    for csv_path in csv_files:
        logger.info(f"Parsing: {csv_path.name}")
        try:
            df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
        except Exception as e:
            logger.warning(f"  ✗ Could not read {csv_path.name}: {e}")
            continue

        df.columns = [c.strip() for c in df.columns]
        cols = df.columns.tolist()

        clip_col  = _find_col(cols, _CLIP_ID_COLS)
        gloss_col = _find_col(cols, _GLOSS_COLS)

        if not clip_col:
            logger.warning(f"  ✗ Could not find clip ID column in {csv_path.name}. "
                           f"Columns: {cols}. "
                           f"Add your column name to _CLIP_ID_COLS in surdobot_parser.py")
            continue
        if not gloss_col:
            logger.warning(f"  ✗ Could not find gloss column in {csv_path.name}. "
                           f"Columns: {cols}. "
                           f"Add your column name to _GLOSS_COLS in surdobot_parser.py")
            continue

        for _, row in df.iterrows():
            total_rows += 1
            clip_id   = str(row[clip_col]).strip()
            gloss_bag = _parse_gloss_cell(row[gloss_col])

            if not gloss_bag:
                logger.debug(f"  Skipping row with empty gloss bag: {clip_id}")
                continue

            video_path = _find_video_file(video_root, clip_id)
            if video_path is None:
                logger.debug(f"  ✗ Video not found for clip_id={clip_id}")
                missing_videos += 1
                continue

            # Probe video for duration
            try:
                meta = probe_video(video_path)
                duration = meta["duration_sec"]
            except Exception:
                duration = 30.0  # fallback assumption

            records.append({
                "clip_id":        clip_id,
                "video_path":     str(video_path),
                "gloss_bag":      gloss_bag,
                "has_audio":      False,   # Surdobot never has audio
                "duration_sec":   duration,
                "source":         "surdobot",
                "split":          None,    # assigned below
            })

    logger.info(f"Parsed {len(records)} valid clips from {total_rows} rows "
                f"({missing_videos} missing video files skipped)")

    # ── Merge multi-annotator duplicates (same clip_id, different annotators) ──
    # Must happen BEFORE split assignment so all annotations of a clip stay together
    records = merge_duplicate_clips(records)
    logger.info(f"After merging duplicate annotations: {len(records)} unique clips")

    # ── Assign train / val / test splits ──────────────────────────────────────
    random.seed(seed)
    random.shuffle(records)
    n = len(records)
    n_train = int(n * split_ratios[0])
    n_val   = int(n * split_ratios[1])

    for i, rec in enumerate(records):
        if i < n_train:
            rec["split"] = "train"
        elif i < n_train + n_val:
            rec["split"] = "val"
        else:
            rec["split"] = "test"

    counts = {s: sum(1 for r in records if r["split"] == s)
              for s in ["train", "val", "test"]}
    logger.info(f"Split counts: {counts}")

    # ── Vocabulary stats ──────────────────────────────────────────────────────
    all_glosses = [g for rec in records for g in rec["gloss_bag"]]
    vocab = sorted(set(all_glosses))
    logger.info(f"Gloss vocabulary size: {len(vocab)}")

    return records


def build_gloss_vocab(records: list[dict], min_freq: int = 3) -> dict:
    """
    Build a gloss-to-index vocabulary from a list of clip records.
    Filters out low-frequency noise and typos below min_freq threshold.
    """
    from collections import Counter
    
    # 1. Count frequencies across the entire dataset
    gloss_counts = Counter()
    for rec in records:
        gloss_counts.update(rec.get("gloss_bag", []))
        
    # 2. Filter out words that don't meet our minimum occurrence threshold
    valid_glosses = sorted([
        word for word, count in gloss_counts.items() if count >= min_freq
    ])
    
    # 3. Establish structural indexing configurations
    gloss2idx = {"<blank>": 0, "<unk>": 1}
    for g in valid_glosses:
        gloss2idx[g] = len(gloss2idx)
        
    idx2gloss = {v: k for k, v in gloss2idx.items()}
    
    # 4. CRUCIAL STEP: Update records to filter out dropped singletons
    # This prevents the records from containing tokens missing from gloss2idx
    for rec in records:
        cleaned_bag = []
        for g in rec["gloss_bag"]:
            if g in gloss2idx:
                cleaned_bag.append(g)
            else:
                # Optional: remap to unk or just drop it if you want strict gloss labels
                if "<unk>" not in cleaned_bag:
                    cleaned_bag.append("<unk>")
        rec["gloss_bag"] = list(dict.fromkeys(cleaned_bag))
        
    print(f"✂️ Dropped {len(gloss_counts) - len(valid_glosses)} rare tokens/typos appearing < {min_freq} times.")
    
    return {
        "gloss2idx": gloss2idx,
        "idx2gloss": idx2gloss,
        "vocab_size": len(gloss2idx),
    }


In [22]:
cfg = load_config()
records = parse_surdobot(
    annotations_dir=cfg.paths.surdobot_annotations,
    video_root=cfg.paths.surdobot_videos,
    split_ratios=(cfg.splits.train, cfg.splits.val, cfg.splits.test),
    seed=cfg.splits.seed,
)
vocab = build_gloss_vocab(records)
print(f"\nTotal clips : {len(records)}")
print(f"Vocab size  : {vocab['vocab_size']}")
print(f"Sample record:\n  {records[0]}")
print(f"Sample glosses: {list(vocab['gloss2idx'].items())[:10]}")

[2026-06-29 16:25:43] INFO      __main__  — Found 16 annotation CSV(s) in /run/media/gygy/T7/KRSL_OnlineSchool/KRSL_OnlineSchool_final_dataset/Surdobot_Annotations
[2026-06-29 16:25:43] INFO      __main__  — Parsing: ._annotations_01.21.csv
[2026-06-29 16:25:43] WARNING   __main__  —   ✗ Could not read ._annotations_01.21.csv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte
[2026-06-29 16:25:43] INFO      __main__  — Parsing: ._annotations_01.22.csv
[2026-06-29 16:25:43] WARNING   __main__  —   ✗ Could not read ._annotations_01.22.csv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte
[2026-06-29 16:25:43] INFO      __main__  — Parsing: annotations_01.21.csv
[2026-06-29 16:25:43] INFO      __main__  — Parsing: annotations_01.22.csv
[2026-06-29 16:25:43] INFO      __main__  — Parsing: annotations_02.21.csv
[2026-06-29 16:25:44] INFO      __main__  — Parsing: annotations_02.22.csv
[2026-06-29 16:25:44] INFO      __main__  — Parsing: annotatio

✂️ Dropped 27429 rare tokens/typos appearing < 3 times.

Total clips : 35301
Vocab size  : 13094
Sample record:
  {'clip_id': '27_10/resized_cropped_20_27_07_16_02-2-of-22.mp4', 'video_path': '/run/media/gygy/T7/KRSL_OnlineSchool/KRSL_OnlineSchool_final_dataset/Surdobot_VideoBase/27_10/resized_cropped_20_27_07_16_02-2-of-22.mp4', 'gloss_bag': ['ПРОШЛО', 'ТЕМА', 'СМОТРИ', 'ПЕРВЫЙ', 'ЗАДАНИЕ', 'РИСУНОК', 'ДАЕТ', 'ТАБЛИЦУ', 'ПУТИ', 'ДВИЖЕНИЕ', 'АВТОМОБИЛЬ', 'КАКОЙ', 'ПУТЬ', 'ЕДЕТ', 'ПЕРВАЯ', 'ВТОРАЯ', 'ДЛИНА', 'ВРЕМЯ', 'БЫЛ', 'ДО', 'ОСТАНОВКА', 'СКОЛЬКО', '<unk>', 'СКОРОСТЬ', 'РАССТОЯНИЕ', 'ДОРОГА', 'О', 'А', 'ОТВЕТ', 'МЕТР', 'СЕКУНДУ', 'СЧИТАТЬ'], 'has_audio': False, 'duration_sec': 30.0, 'source': 'surdobot', 'split': 'train'}
Sample glosses: [('<blank>', 0), ('<unk>', 1), ('A', 2), ('B', 3), ('C', 4), ('F', 5), ('G', 6), ('GOOGLE', 7), ('I', 8), ('K', 9)]


In [23]:
import json

records_json_path = "surdobot_dataset_records.json"
with open(records_json_path, "w", encoding="utf-8") as f:
    # indent=4 makes the file human-readable; ensure_ascii=False keeps Russian characters intact
    json.dump(records, f, indent=4, ensure_ascii=False)
print(f"Successfully saved {len(records)} clip records to {records_json_path}")

# 2. Save the vocabulary tracking dictionary
vocab_json_path = "surdobot_vocab.json"
with open(vocab_json_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, indent=4, ensure_ascii=False)
print(f"Successfully saved vocabulary ({vocab['vocab_size']} tokens) to {vocab_json_path}")

Successfully saved 35301 clip records to surdobot_dataset_records.json
Successfully saved vocabulary (13094 tokens) to surdobot_vocab.json


In [24]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path


def load_dataset_files(vocab_path: str, records_path: str):
    """Loads the saved JSON artifacts from disk."""
    print(f"Loading {vocab_path} and {records_path}...")
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)
    with open(records_path, "r", encoding="utf-8") as f:
        records = json.load(f)
    return vocab, records


def run_vocab_diagnostics(vocab: dict, records: list):
    """Runs data integrity diagnostics on the vocabulary and structural text data."""
    print("\n" + "=" * 50)
    print("        SURDOBOT VOCABULARY HEALTH REPORT       ")
    print("=" * 50)

    # 1. Compute Exact Word Frequencies across the entire dataset
    # (Since records store gloss_bag as lists, we count occurrences across all bags)
    gloss_counts = Counter()
    for rec in records:
        gloss_counts.update(rec.get("gloss_bag", []))

    total_tokens = sum(gloss_counts.values())
    unique_tokens_in_data = len(gloss_counts)

    print(f"Total Unique Tokens in Vocab Config : {vocab['vocab_size']}")
    print(f"Total Unique Tokens observed in Data: {unique_tokens_in_data}")
    print(f"Total Token Occurrences across Dataset: {total_tokens}")
    print("-" * 50)

    # 2. Flag Singletons (Words appearing exactly once)
    singletons = [word for word, count in gloss_counts.items() if count == 1]
    low_freq = [
        word for word, count in gloss_counts.items() if 1 < count <= 5
    ]

    print(f"⚠️ Singletons (appear exactly 1 time) : {len(singletons)}")
    print(f"⚠️ Low Frequency (appear 2-5 times)   : {len(low_freq)}")
    if singletons:
        print(f"   Sample singletons: {singletons[:15]}")

    print("-" * 50)

    # 3. Identify Script & Alphabetic Garbage Anomalies
    non_cyrillic = []
    mixed_script = []

    # Regex patterns for validation
    cyrillic_pattern = re.compile(r"^[\u0400-\u04FF\s]+$")
    latin_pattern = re.compile(r"[A-Z]")
    cyrillic_char_pattern = re.compile(r"[\u0400-\u04FF]")

    # Check against keys in gloss2idx (excluding baseline special tokens)
    for word in vocab["gloss2idx"].keys():
        if word in ["<blank>", "<unk>"]:
            continue

        is_pure_cyrillic = bool(cyrillic_pattern.match(word))
        has_latin = bool(latin_pattern.search(word))
        has_cyrillic = bool(cyrillic_char_pattern.search(word))

        if not is_pure_cyrillic and not has_latin:
            non_cyrillic.append(word)
        elif has_latin and has_cyrillic:
            mixed_script.append(word)

    print(f"❌ Purely Non-Cyrillic/Foreign entries: {len(non_cyrillic)}")
    if non_cyrillic:
        print(f"   Sample non-cyrillic: {non_cyrillic[:10]}")

    print(f"❌ Mixed-Script Corruption (Latin + Cyrillic): {len(mixed_script)}")
    if mixed_script:
        print(f"   Sample mixed-script: {mixed_script[:10]}")

    print("-" * 50)

    # 4. Cluster Inflectional Variants by Shared Prefix (Stemming heuristic)
    # Russian adjectives/nouns share long prefixes (e.g., ПЕРВЫЙ/ПЕРВАЯ/ПЕРВОЕ)
    prefix_groups = defaultdict(list)
    min_prefix_len = 4  # Track variations matching the first 4+ characters

    for word in gloss_counts.keys():
        if len(word) >= min_prefix_len:
            stem = word[:min_prefix_len]
            prefix_groups[stem].append(word)

    # Filter out stems that only produced a single word format
    inflection_clusters = {
        stem: words for stem, words in prefix_groups.items() if len(words) > 1
    }

    print(
        f"🔄 Potential Inflectional Variant Clusters: {len(inflection_clusters)}"
    )

    # Display some prominent inflection clusters sorted by variant count
    sorted_clusters = sorted(
        inflection_clusters.items(), key=lambda x: len(x[1]), reverse=True
    )
    for stem, variants in sorted_clusters[:8]:
        # Show total aggregated frequencies inside the cluster variant
        variant_details = [f"{v} ({gloss_counts[v]}x)" for v in variants]
        print(f"   Stem Group [{stem}...]: {', '.join(variant_details)}")

    print("=" * 50)


# --- Execution Hook ---
if __name__ == "__main__":
    # Point these to the files generated by your previous script execution
    VOCAB_FILE = "surdobot_vocab.json"
    RECORDS_FILE = "surdobot_dataset_records.json"

    if Path(VOCAB_FILE).exists() and Path(RECORDS_FILE).exists():
        vocab_data, records_data = load_dataset_files(VOCAB_FILE, RECORDS_FILE)
        run_vocab_diagnostics(vocab_data, records_data)
    else:
        print(
            f"Error: Missing targeted target files. Verify {VOCAB_FILE} and {RECORDS_FILE} exist."
        )

Loading surdobot_vocab.json and surdobot_dataset_records.json...

        SURDOBOT VOCABULARY HEALTH REPORT       
Total Unique Tokens in Vocab Config : 13094
Total Unique Tokens observed in Data: 13093
Total Token Occurrences across Dataset: 880027
--------------------------------------------------
⚠️ Singletons (appear exactly 1 time) : 0
⚠️ Low Frequency (appear 2-5 times)   : 4722
--------------------------------------------------
❌ Purely Non-Cyrillic/Foreign entries: 10
   Sample non-cyrillic: ['А.Б', 'В.П', 'Д.П', 'И.П', 'КВ.М', 'МИР(ПЛАНЕТА', 'П.П', 'ПОГОДА(ПРИРОДА', 'Р.П', 'Т.П']
❌ Mixed-Script Corruption (Latin + Cyrillic): 0
--------------------------------------------------
🔄 Potential Inflectional Variant Clusters: 1870
   Stem Group [ПЕРЕ...]: ПЕРЕЕХАТЬ (157x), ПЕРЕДАТЬ (300x), ПЕРЕХОДИМ (11x), ПЕРЕ (6x), ПЕРЕВОДИТЬ (34x), ПЕРЕСТАВИТЬ (29x), ПЕРЕЙТИ (56x), ПЕРЕВОДИТСЯ (19x), ПЕРЕДАЛИ (32x), ПЕРЕХОДИТЬ (63x), ПЕРЕХОДИТСЯ (3x), ПЕРЕПИСКИ (3x), ПЕРЕЦ (4x), ПЕРЕД (194x), ПЕРЕ

In [1]:
"""
Keypoint Loading Validation — Phase 1 sanity check

Run this BEFORE running feature extraction at scale. It checks, on a small
sample of real clips from the parsed Surdobot manifest:

  1. Does the video → .npy filename mapping actually resolve?
     (Surdobot videos live in subfolders like "27_10/clip.mp4" — does
     hs_labeled/ expect just the stem "clip.npy" with no subfolder, or
     does it mirror the subfolder structure?)
  2. What shape do the loaded arrays actually have?
     ((T, 42, 2), (T, 84) flat, or something else entirely)
  3. Does the validity-mask logic correctly separate "hand missing"
     (all-zero block) from real wrist-at-origin data on real frames?

This is READ-ONLY — it does not modify any files or write any output
besides the diagnostic printout.
"""

import random
from pathlib import Path

import numpy as np

from load_config import load_config
from common import get_logger, load_manifest, compute_validity_mask

logger = get_logger(__name__)


def find_npy_candidates(video_path: str, hs_labeled_dir: Path) -> list[Path]:
    """
    Try several plausible filename conventions for a given video path,
    and return ALL candidates that actually exist on disk — so we can see
    which convention (if any) is correct, rather than assuming one.
    """
    video_path = Path(video_path)
    stem = video_path.stem  # e.g. "resized_cropped_20_27_07_16_02-2-of-22"

    candidates = {
        "flat_same_stem":        hs_labeled_dir / f"{stem}.npy",
        "mirrored_subfolder":     hs_labeled_dir / video_path.parent.name / f"{stem}.npy",
        "flat_with_parent_prefix": hs_labeled_dir / f"{video_path.parent.name}_{stem}.npy",
    }

    found = []
    for label, path in candidates.items():
        if path.exists():
            found.append((label, path))

    return found


def inspect_npy_file(path: Path) -> dict:
    """Load a .npy file and report its shape, dtype, and value ranges."""
    arr = np.load(str(path))
    info = {
        "path":       str(path),
        "shape":      arr.shape,
        "dtype":      str(arr.dtype),
        "min":        float(arr.min()) if arr.size else None,
        "max":        float(arr.max()) if arr.size else None,
        "n_nan":      int(np.isnan(arr).sum()) if np.issubdtype(arr.dtype, np.floating) else 0,
    }
    return info, arr


def run_validation(manifest_path: str | Path, hs_labeled_dir: str | Path,
                   n_samples: int = 8, seed: int = 42) -> None:
    hs_labeled_dir = Path(hs_labeled_dir)
    records = load_manifest(manifest_path)

    if not hs_labeled_dir.exists():
        logger.error(f"hs_labeled directory does not exist: {hs_labeled_dir}")
        return

    # ── Quick scan: what does hs_labeled/ actually contain? ──────────────────────
    npy_files = list(hs_labeled_dir.glob("**/*.npy"))
    logger.info(f"Found {len(npy_files)} total .npy files under {hs_labeled_dir}")
    if npy_files:
        logger.info(f"  Sample filenames: {[p.name for p in npy_files[:5]]}")
        subdirs = {p.parent for p in npy_files}
        logger.info(f"  Files span {len(subdirs)} distinct subdirectories "
                   f"(1 means flat, >1 means mirrored structure)")

    # ── Sample random clips from the manifest ────────────────────────────────────
    random.seed(seed)
    sample = random.sample(records, min(n_samples, len(records)))

    n_resolved, n_unresolved = 0, 0
    convention_votes = {}

    logger.info(f"\n{'='*70}")
    logger.info(f"Testing {len(sample)} sample clips against hs_labeled/")
    logger.info(f"{'='*70}")

    for rec in sample:
        clip_id    = rec["clip_id"]
        video_path = rec["video_path"]

        logger.info(f"\n── {clip_id} ──")
        found = find_npy_candidates(video_path, hs_labeled_dir)

        if not found:
            logger.warning(f"  ✗ NO MATCH under any tested naming convention")
            n_unresolved += 1
            continue

        n_resolved += 1
        for label, path in found:
            convention_votes[label] = convention_votes.get(label, 0) + 1
            logger.info(f"  ✓ Matched convention '{label}': {path}")

            try:
                info, arr = inspect_npy_file(path)
                logger.info(f"    shape={info['shape']}  dtype={info['dtype']}  "
                           f"range=[{info['min']:.4f}, {info['max']:.4f}]  "
                           f"nan_count={info['n_nan']}")

                # ── Shape sanity check ────────────────────────────────────────────
                if arr.ndim == 3 and arr.shape[1:] == (42, 2):
                    logger.info(f"    ✓ Shape matches expected (T, 42, 2)")
                    coords = arr
                elif arr.ndim == 2 and arr.shape[1] == 84:
                    logger.info(f"    ✓ Shape matches expected flat (T, 84) → reshapeable to (T,42,2)")
                    coords = arr.reshape(arr.shape[0], 42, 2)
                else:
                    logger.warning(f"    ⚠ UNEXPECTED SHAPE — does not match (T,42,2) or (T,84). "
                                  f"Inspect manually.")
                    continue

                # ── Validity mask check ───────────────────────────────────────────
                validity = compute_validity_mask(coords)
                pct_left_valid  = 100 * validity[:, 0].mean()
                pct_right_valid = 100 * validity[:, 1].mean()
                logger.info(f"    Left hand valid:  {pct_left_valid:.1f}% of frames")
                logger.info(f"    Right hand valid: {pct_right_valid:.1f}% of frames")

                if pct_left_valid < 1 and pct_right_valid < 1:
                    logger.warning(f"    ⚠ BOTH hands almost always invalid — check if "
                                  f"validity-mask logic or the data itself is wrong")

            except Exception as e:
                logger.error(f"    ✗ Failed to load/inspect: {e}")

    # ── Summary ───────────────────────────────────────────────────────────────────
    logger.info(f"\n{'='*70}")
    logger.info(f"SUMMARY")
    logger.info(f"{'='*70}")
    logger.info(f"Resolved:   {n_resolved}/{len(sample)}")
    logger.info(f"Unresolved: {n_unresolved}/{len(sample)}")
    logger.info(f"Naming convention votes: {convention_votes}")

    if n_unresolved > 0:
        logger.warning(
            f"\n⚠ {n_unresolved} clips had NO matching .npy file under any tested "
            f"convention. Either: (a) keypoint extraction is incomplete for these "
            f"clips, (b) the naming convention differs from what was tested here — "
            f"inspect the actual hs_labeled/ filenames vs. video filenames manually, "
            f"or (c) these specific clips were skipped during keypoint extraction."
        )

    if len(set(convention_votes.keys())) > 1:
        logger.warning(
            f"\n⚠ Multiple naming conventions matched across different clips — "
            f"this is suspicious. Verify these aren't false-positive matches "
            f"(e.g. two different clips coincidentally having a file at both paths)."
        )
    elif convention_votes:
        winning = max(convention_votes, key=convention_votes.get)
        logger.info(f"\n✓ Consistent naming convention detected: '{winning}'")
        logger.info(f"  Use this convention in load_keypoints_for_clip() / video_to_npy_path()")


ImportError: cannot import name 'compute_validity_mask' from 'common' (/home/gygy/Desktop/data-annotation/data-annotation-aiym/notebooks/common.py)

In [1]:
import numpy as np


def compute_validity_mask(coords: np.ndarray) -> np.ndarray:
    """
    Determine, per frame and per hand, whether that hand's keypoints are
    real detections or a "missing hand" placeholder.

    Parameters
    ----------
    coords : np.ndarray, shape (T, 2, 21, 2)
        Keypoints for T frames. Axis 1 is the hand (0=left, 1=right),
        axis 2 is the 21 landmarks, axis 3 is (x, y).

    Returns
    -------
    validity : np.ndarray, shape (T, 2), dtype=bool
        validity[:, 0] -> left hand valid per frame
        validity[:, 1] -> right hand valid per frame

    Notes
    -----
    A hand is marked INVALID for a frame only when *every* one of its
    21 (x, y) points is exactly zero — i.e. the whole block was zero-
    filled by the extractor because the hand wasn't detected.

    This deliberately does NOT flag a hand as invalid just because a
    single landmark (e.g. the wrist, index 0) happens to be at (0, 0)
    on a real, otherwise-populated frame — that's legitimate data, not
    a missing-hand placeholder. Only an all-zero block counts as missing.
    """
    if coords.ndim != 4 or coords.shape[1] != 2 or coords.shape[2] != 21 or coords.shape[3] != 2:
        raise ValueError(
            f"Expected coords of shape (T, 2, 21, 2), got {coords.shape}"
        )

    # A hand is "missing" for a frame iff ALL 21*2 = 42 values for that
    # hand are zero. Reduce over the landmark axis (2) and coord axis (3).
    all_zero = np.all(coords == 0, axis=(2, 3))  # (T, 2) -> True where missing

    validity = ~all_zero  # (T, 2), True where valid
    return validity

In [5]:
"""
Build manifest.json by walking the raw Surdobot video directory.

Produces a JSON list of {"clip_id": ..., "video_path": ...} records,
which is what run_validation() / load_manifest() expect.
"""

import json
from pathlib import Path


def build_manifest(video_root: str | Path, output_path: str | Path,
                    extensions: tuple[str, ...] = (".mp4", ".mov", ".avi", ".mkv")) -> None:
    video_root = Path(video_root)
    output_path = Path(output_path)

    if not video_root.exists():
        raise FileNotFoundError(f"Video root does not exist: {video_root}")

    video_files = [
        p for p in video_root.glob("**/*")
        if p.suffix.lower() in extensions
    ]

    print(f"Found {len(video_files)} video files under {video_root}")

    records = []
    for p in sorted(video_files):
        rel_path = p.relative_to(video_root)
        records.append({
            "clip_id": p.stem,
            "video_path": str(rel_path),  # relative, matches how find_npy_candidates uses .parent.name
        })

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    print(f"Wrote {len(records)} records to {output_path}")

In [6]:
build_manifest(
    video_root="/run/media/gygy/T7/KRSL_OnlineSchool/KRSL_OnlineSchool_final_dataset/Surdobot_VideoBase",  # <-- update this
    output_path="/run/media/gygy/T7/KRSL_OnlineSchool/validation/manifest.json",
)

Found 39332 video files under /run/media/gygy/T7/KRSL_OnlineSchool/KRSL_OnlineSchool_final_dataset/Surdobot_VideoBase
Wrote 39332 records to /run/media/gygy/T7/KRSL_OnlineSchool/validation/manifest.json


In [7]:
"""
Keypoint Loading Validation — Phase 1 sanity check

Run this BEFORE running feature extraction at scale. It checks, on a small
sample of real clips from the parsed Surdobot manifest:

  1. Does the video → .npy filename mapping actually resolve?
     (Surdobot videos live in subfolders like "27_10/clip.mp4" — does
     hs_labeled/ expect just the stem "clip.npy" with no subfolder, or
     does it mirror the subfolder structure?)
  2. What shape do the loaded arrays actually have?
     ((T, 2, 21, 2), (T, 42, 2), (T, 84) flat, or something else entirely)
  3. Does the validity-mask logic correctly separate "hand missing"
     (all-zero block) from real wrist-at-origin data on real frames?

This is READ-ONLY — it does not modify any files or write any output
besides the diagnostic printout.
"""

import random
from pathlib import Path

import numpy as np

from load_config import load_config
from common import get_logger, load_manifest
logger = get_logger(__name__)


def find_npy_candidates(video_path: str, hs_labeled_dir: Path) -> list[Path]:
    """
    Try several plausible filename conventions for a given video path,
    and return ALL candidates that actually exist on disk — so we can see
    which convention (if any) is correct, rather than assuming one.
    """
    video_path = Path(video_path)
    stem = video_path.stem  # e.g. "resized_cropped_20_27_07_16_02-2-of-22"

    candidates = {
        "flat_same_stem":        hs_labeled_dir / f"{stem}.npy",
        "mirrored_subfolder":     hs_labeled_dir / video_path.parent.name / f"{stem}.npy",
        "flat_with_parent_prefix": hs_labeled_dir / f"{video_path.parent.name}_{stem}.npy",
    }

    found = []
    for label, path in candidates.items():
        if path.exists():
            found.append((label, path))

    return found


def inspect_npy_file(path: Path) -> dict:
    """Load a .npy file and report its shape, dtype, and value ranges."""
    arr = np.load(str(path))
    info = {
        "path":       str(path),
        "shape":      arr.shape,
        "dtype":      str(arr.dtype),
        "min":        float(arr.min()) if arr.size else None,
        "max":        float(arr.max()) if arr.size else None,
        "n_nan":      int(np.isnan(arr).sum()) if np.issubdtype(arr.dtype, np.floating) else 0,
    }
    return info, arr


def normalize_to_coords(arr: np.ndarray, logger) -> np.ndarray | None:
    """
    Normalize whatever shape convention the .npy file uses into the
    canonical (T, 2, 21, 2) layout expected by compute_validity_mask.

    Returns None if the shape doesn't match any known convention.
    """
    if arr.ndim == 4 and arr.shape[1:] == (2, 21, 2):
        logger.info(f"    ✓ Shape matches expected (T, 2, 21, 2)")
        return arr

    if arr.ndim == 3 and arr.shape[1:] == (42, 2):
        logger.info(f"    ✓ Shape matches (T, 42, 2) → reshaping to (T, 2, 21, 2)")
        # Assumes first 21 rows = left hand, next 21 = right hand
        return arr.reshape(arr.shape[0], 2, 21, 2)

    if arr.ndim == 2 and arr.shape[1] == 84:
        logger.info(f"    ✓ Shape matches flat (T, 84) → reshaping to (T, 2, 21, 2)")
        return arr.reshape(arr.shape[0], 2, 21, 2)

    logger.warning(f"    ⚠ UNEXPECTED SHAPE {arr.shape} — does not match any known "
                  f"convention (T,2,21,2), (T,42,2), (T,84). Inspect manually.")
    return None


def run_validation(manifest_path: str | Path, hs_labeled_dir: str | Path,
                   n_samples: int = 8, seed: int = 42) -> None:
    hs_labeled_dir = Path(hs_labeled_dir)
    records = load_manifest(manifest_path)

    if not hs_labeled_dir.exists():
        logger.error(f"hs_labeled directory does not exist: {hs_labeled_dir}")
        return

    # ── Quick scan: what does hs_labeled/ actually contain? ──────────────────────
    npy_files = list(hs_labeled_dir.glob("**/*.npy"))
    logger.info(f"Found {len(npy_files)} total .npy files under {hs_labeled_dir}")
    if npy_files:
        logger.info(f"  Sample filenames: {[p.name for p in npy_files[:5]]}")
        subdirs = {p.parent for p in npy_files}
        logger.info(f"  Files span {len(subdirs)} distinct subdirectories "
                   f"(1 means flat, >1 means mirrored structure)")

    # ── Sample random clips from the manifest ────────────────────────────────────
    random.seed(seed)
    sample = random.sample(records, min(n_samples, len(records)))

    n_resolved, n_unresolved = 0, 0
    convention_votes = {}

    logger.info(f"\n{'='*70}")
    logger.info(f"Testing {len(sample)} sample clips against hs_labeled/")
    logger.info(f"{'='*70}")

    for rec in sample:
        clip_id    = rec["clip_id"]
        video_path = rec["video_path"]

        logger.info(f"\n── {clip_id} ──")
        found = find_npy_candidates(video_path, hs_labeled_dir)

        if not found:
            logger.warning(f"  ✗ NO MATCH under any tested naming convention")
            n_unresolved += 1
            continue

        n_resolved += 1
        for label, path in found:
            convention_votes[label] = convention_votes.get(label, 0) + 1
            logger.info(f"  ✓ Matched convention '{label}': {path}")

            try:
                info, arr = inspect_npy_file(path)
                logger.info(f"    shape={info['shape']}  dtype={info['dtype']}  "
                           f"range=[{info['min']:.4f}, {info['max']:.4f}]  "
                           f"nan_count={info['n_nan']}")

                # ── Shape sanity check / normalization ────────────────────────────
                coords = normalize_to_coords(arr, logger)
                if coords is None:
                    continue

                # ── Validity mask check ───────────────────────────────────────────
                validity = compute_validity_mask(coords)
                pct_left_valid  = 100 * validity[:, 0].mean()
                pct_right_valid = 100 * validity[:, 1].mean()
                logger.info(f"    Left hand valid:  {pct_left_valid:.1f}% of frames")
                logger.info(f"    Right hand valid: {pct_right_valid:.1f}% of frames")

                if pct_left_valid < 1 and pct_right_valid < 1:
                    logger.warning(f"    ⚠ BOTH hands almost always invalid — check if "
                                  f"validity-mask logic or the data itself is wrong")

            except Exception as e:
                logger.error(f"    ✗ Failed to load/inspect: {e}")

    # ── Summary ───────────────────────────────────────────────────────────────────
    logger.info(f"\n{'='*70}")
    logger.info(f"SUMMARY")
    logger.info(f"{'='*70}")
    logger.info(f"Resolved:   {n_resolved}/{len(sample)}")
    logger.info(f"Unresolved: {n_unresolved}/{len(sample)}")
    logger.info(f"Naming convention votes: {convention_votes}")

    if n_unresolved > 0:
        logger.warning(
            f"\n⚠ {n_unresolved} clips had NO matching .npy file under any tested "
            f"convention. Either: (a) keypoint extraction is incomplete for these "
            f"clips, (b) the naming convention differs from what was tested here — "
            f"inspect the actual hs_labeled/ filenames vs. video filenames manually, "
            f"or (c) these specific clips were skipped during keypoint extraction."
        )

    if len(set(convention_votes.keys())) > 1:
        logger.warning(
            f"\n⚠ Multiple naming conventions matched across different clips — "
            f"this is suspicious. Verify these aren't false-positive matches "
            f"(e.g. two different clips coincidentally having a file at both paths)."
        )
    elif convention_votes:
        winning = max(convention_votes, key=convention_votes.get)
        logger.info(f"\n✓ Consistent naming convention detected: '{winning}'")
        logger.info(f"  Use this convention in load_keypoints_for_clip() / video_to_npy_path()")

In [8]:
run_validation(
    manifest_path="/run/media/gygy/T7/KRSL_OnlineSchool/validation/manifest.json",  # or .csv — whatever load_manifest expects
    hs_labeled_dir="/run/media/gygy/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints",
    n_samples=8,
    seed=42,
)

[2026-07-08 16:20:07] INFO      __main__  — Found 32981 total .npy files under /run/media/gygy/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints
[2026-07-08 16:20:07] INFO      __main__  —   Sample filenames: ['resized_cropped_09_12_10_10_01-1-of-29.npy', 'resized_cropped_09_12_10_10_01-10-of-29.npy', 'resized_cropped_09_12_10_10_01-16-of-29.npy', 'resized_cropped_09_12_10_10_01-18-of-29.npy', 'resized_cropped_09_12_10_10_01-26-of-29.npy']
[2026-07-08 16:20:07] INFO      __main__  —   Files span 1 distinct subdirectories (1 means flat, >1 means mirrored structure)
[2026-07-08 16:20:07] INFO      __main__  — 
[2026-07-08 16:20:07] INFO      __main__  — Testing 8 sample clips against hs_labeled/
[2026-07-08 16:20:07] INFO      __main__  — ======================================================================
[2026-07-08 16:20:07] INFO      __main__  — 
── resized_cropped_02_12_11_12_01-3-of-28 ──
[2026-07-08 16:20:07] INFO      __main__  —   ✓ Matched convention 'flat_same_stem': /run/media

In [9]:
"""
Dataset Coverage Check — Phase 1 pre-flight, before VideoMAE

Cheap, read-only sanity check on surdobot_dataset_records.json.
Verifies, per record:
  1. video_path exists on disk
  2. a corresponding keypoint .npy file exists (flat_same_stem convention,
     confirmed earlier: hs_labeled_dir / f"{stem}.npy")
  3. gloss_bag is non-empty

No model loading, no GPU. Prints a summary + a breakdown of exactly
which check(s) are failing for problem records.
"""

import json
from pathlib import Path

RECORDS_PATH = "/run/media/gygy/T7/KRSL_OnlineSchool/validation/surdobot_dataset_records.json"
HS_LABELED_DIR = Path("/run/media/gygy/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints")


def check_coverage(records_path: str | Path, hs_labeled_dir: Path) -> None:
    with open(records_path, encoding="utf-8") as f:
        records = json.load(f)

    total = len(records)
    missing_video = []
    missing_npy = []
    empty_gloss = []

    for rec in records:
        clip_id = rec.get("clip_id", "<no clip_id>")
        video_path = Path(rec["video_path"])
        npy_path = hs_labeled_dir / f"{video_path.stem}.npy"
        gloss_bag = rec.get("gloss_bag", [])

        if not video_path.exists():
            missing_video.append(clip_id)
        if not npy_path.exists():
            missing_npy.append(clip_id)
        if not gloss_bag:
            empty_gloss.append(clip_id)

    n_bad = len(set(missing_video) | set(missing_npy) | set(empty_gloss))
    n_clean = total - n_bad

    print(f"{'='*60}")
    print(f"COVERAGE SUMMARY")
    print(f"{'='*60}")
    print(f"Total records:            {total}")
    print(f"Clean (pass all 3):       {n_clean}  ({100*n_clean/total:.1f}%)")
    print(f"Missing video file:       {len(missing_video)}")
    print(f"Missing keypoint .npy:    {len(missing_npy)}")
    print(f"Empty gloss_bag:          {len(empty_gloss)}")

    for label, ids in [("MISSING VIDEO", missing_video),
                        ("MISSING NPY", missing_npy),
                        ("EMPTY GLOSS BAG", empty_gloss)]:
        if ids:
            print(f"\n{label} — {len(ids)} clips, first 10:")
            for cid in ids[:10]:
                print(f"  {cid}")

In [10]:
check_coverage(RECORDS_PATH, HS_LABELED_DIR)

COVERAGE SUMMARY
Total records:            35301
Clean (pass all 3):       30235  (85.6%)
Missing video file:       0
Missing keypoint .npy:    5066
Empty gloss_bag:          0

MISSING NPY — 5066 clips, first 10:
  2020/resized_cropped_22_12_5_05_02-23-of-31.mp4
  24_11/resized_cropped_23_09_1_01_02-16-of-17.mp4
  2020/resized_cropped_17_11_7_15_02-13-of-22.mp4
  24_11/resized_cropped_23_09_9_08_01-19-of-24.mp4
  24_11/resized_cropped_07_10_7_03_03-17-of-22.mp4
  24_11/resized_cropped_24_11_1_01_03-7-of-10.mp4
  2020/resized_cropped_17_11_10_09_01-27-of-30.mp4
  24_11/resized_cropped_06_10_10_14_01-2-of-24.mp4
  24_11/resized_cropped_12_10_3_04_02-1-of-21.mp4
  24_11/resized_cropped_26_11_4_02_03-14-of-20.mp4


In [11]:
import json
from pathlib import Path

RECORDS_PATH = "/run/media/gygy/T7/KRSL_OnlineSchool/validation/surdobot_dataset_records.json"
HS_LABELED_DIR = Path("/run/media/gygy/T7/KRSL_OnlineSchool/surdobot_raw_hs_keypoints")

with open(RECORDS_PATH, encoding="utf-8") as f:
    records = json.load(f)

missing_npy_no_gloss = []   # missing npy AND no annotation -> safe to drop
missing_npy_has_gloss = []  # missing npy BUT has annotation -> think twice before dropping

for rec in records:
    video_path = Path(rec["video_path"])
    npy_path = HS_LABELED_DIR / f"{video_path.stem}.npy"
    gloss_bag = rec.get("gloss_bag", [])

    if not npy_path.exists():
        if not gloss_bag:
            missing_npy_no_gloss.append(rec["clip_id"])
        else:
            missing_npy_has_gloss.append(rec["clip_id"])

print(f"Missing npy AND empty gloss_bag (safe to drop): {len(missing_npy_no_gloss)}")
print(f"Missing npy BUT has gloss_bag (annotated, just no keypoints): {len(missing_npy_has_gloss)}")

Missing npy AND empty gloss_bag (safe to drop): 0
Missing npy BUT has gloss_bag (annotated, just no keypoints): 5066
